# Kaggle Playground Series s6e10: TabM (Tabular BatchEnsemble MLP)
**Predicting Airline Satisfaction**

In this notebook, we train **TabM** (*Tabular Deep Learning with Parameter-Efficient Ensembling*, ICLR 2025 by Yandex Research) on the Kaggle Playground Series s6e10 dataset.

### Why TabM?
- **Parameter-Efficient Ensembling**: Rather than training a single standard MLP or multiple independent heavy neural networks, TabM uses a BatchEnsemble formulation to simulate an ensemble of $k$ submodels (default $k=32$) within a single unified model.
- **GBDT-Competitive**: TabM is specifically designed for tabular data and performs competitively with top gradient-boosted decision tree algorithms (LightGBM, XGBoost, CatBoost).
- **GPU Acceleration**: TabM fully leverages GPU tensor acceleration, making multi-epoch training across ~700,000 rows fast and scalable.
- **Evaluation Metric**: Area Under the ROC Curve (**ROC-AUC**).

In [ ]:
# Install TabM package
# Kaggle notebooks have internet access enabled, so this installs in a few seconds.
!pip install -q tabm

In [ ]:
import os
import gc
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import roc_auc_score, log_loss, roc_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tabm import TabM

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

# Hardware Acceleration Check: GPU is strongly recommended for TabM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. For ~700k rows, running on GPU is strongly recommended.")

## Configuration
All paths, hyperparameters, and settings are defined in a centralized configuration class.

In [ ]:
class CFG:
    competition = 'playground-series-s6e10'
    target = 'satisfaction'
    id_col = 'id'
    seed = 42
    n_splits = 5
    
    # TabM Architecture
    k = 32               # Number of virtual ensemble submodels
    n_blocks = 3         # Number of MLP blocks
    d_block = 512        # Hidden dimension per block
    dropout = 0.1        # Dropout rate
    
    # Training
    batch_size = 2048    # Batch size (2048 is fast and memory-efficient on GPU)
    epochs = 20          # Max epochs per fold
    lr = 1e-3            # Initial learning rate
    weight_decay = 1e-4  # Weight decay for AdamW
    patience = 4         # Early stopping patience on validation ROC-AUC

## Data Loading
We dynamically detect the dataset directory so that this notebook works seamlessly on Kaggle servers as well as in local environments.

In [ ]:
# Locate dataset directory
if os.path.exists(f'/kaggle/input/{CFG.competition}'):
    DATA_DIR = Path(f'/kaggle/input/{CFG.competition}')
elif os.path.exists(f'/kaggle/input/competitions/{CFG.competition}'):
    DATA_DIR = Path(f'/kaggle/input/competitions/{CFG.competition}')
else:
    DATA_DIR = Path('../../dataset')

print(f"Data Directory: {DATA_DIR}")

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f"Train samples: {len(train_df):,} | Columns: {train_df.shape[1]}")
print(f"Test samples:  {len(test_df):,} | Columns: {test_df.shape[1]}")
display(train_df.head(3))

## Feature Engineering & Preprocessing
The dataset comprises:
1. **Categorical Features**: `Gender`, `Customer Type`, `Type of Travel`, `Class`.
2. **Continuous Features**: `Age`, `Flight Distance`, `Departure Delay in Minutes`, `Arrival Delay in Minutes`.
3. **Survey Ratings (1-5)**: 13 satisfaction ratings covering online booking, food, wifi, comfort, cleanliness, and service.

### Feature Transformations:
- **Delay Imputation & Features**: Missing values in `Arrival Delay in Minutes` are imputed using `Departure Delay in Minutes` (correlation > 0.84). We also compute `Total_Delay`, `Delay_Diff`, and `Is_Delayed`.
- **Survey Aggregations**: We extract statistical summaries across survey ratings (`Rating_Mean`, `Rating_Sum`, `Rating_Std`, `Rating_Min`, `Rating_Max`) and domain composites (`Comfort_Score`, `Service_Score`, `Convenience_Score`).
- **Encoding**: Categorical features are converted to integer indices (0 to cardinality - 1) for TabM's native categorical embeddings via `cat_cardinalities`. Continuous features are normalized with `StandardScaler`.

In [ ]:
survey_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness'
]

cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

def engineer_features(df_train, df_test):
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    for df in [df_train, df_test]:
        # Impute missing Arrival Delay with Departure Delay
        df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])
        
        # Delay features
        df['Total_Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']
        df['Delay_Diff'] = df['Arrival Delay in Minutes'] - df['Departure Delay in Minutes']
        df['Is_Delayed'] = ((df['Departure Delay in Minutes'] > 0) | (df['Arrival Delay in Minutes'] > 0)).astype(float)
        
        # Survey summary statistics
        df['Rating_Mean'] = df[survey_cols].mean(axis=1)
        df['Rating_Sum'] = df[survey_cols].sum(axis=1)
        df['Rating_Std'] = df[survey_cols].std(axis=1)
        df['Rating_Min'] = df[survey_cols].min(axis=1)
        df['Rating_Max'] = df[survey_cols].max(axis=1)
        
        # Domain composite scores
        df['Comfort_Score'] = df[['Seat comfort', 'Food and drink', 'Cleanliness', 'Leg room service']].mean(axis=1)
        df['Service_Score'] = df[['Inflight wifi service', 'Inflight entertainment', 'Online boarding',
                                  'On-board service', 'Baggage handling', 'Checkin service']].mean(axis=1)
        df['Convenience_Score'] = df[['Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location']].mean(axis=1)
        
    return df_train, df_test

train_eng, test_eng = engineer_features(train_df, test_df)

# Target & Test IDs
y = train_eng[CFG.target].astype(int).values
test_ids = test_eng[CFG.id_col].values

# Identify numeric columns
feature_cols = [c for c in train_eng.columns if c not in [CFG.id_col, CFG.target]]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")

# Encode Categorical Columns as integers
cat_encoder = OrdinalEncoder(dtype=np.int64)
train_eng[cat_cols] = cat_encoder.fit_transform(train_eng[cat_cols])
test_eng[cat_cols] = cat_encoder.transform(test_eng[cat_cols])

# Cardinalities for TabM
cat_cardinalities = [len(categories) for categories in cat_encoder.categories_]
print(f"Categorical cardinalities: {cat_cardinalities}")

## PyTorch Dataset
A standard PyTorch `Dataset` that yields continuous tensor inputs `x_num`, categorical integer indices `x_cat`, and optional target `y`.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.x_num[idx], self.x_cat[idx], self.y[idx]
        return self.x_num[idx], self.x_cat[idx]

## TabM Loss & Prediction Utilities
### Training Loss Formulation
TabM outputs $k$ predictions per object (shape `(batch_size, k, 1)`). To properly train the parameter-efficient ensemble, **the mean loss across the $k$ submodels must be optimized independently**:
$$\mathcal{L} = \frac{1}{k} \sum_{i=1}^k \text{BCE}(\hat{y}_i, y)$$

### Inference Ensemble Averaging
During validation and inference, we compute predicted probabilities by applying the sigmoid activation and averaging across all $k$ submodels:
$$P(y=1) = \frac{1}{k} \sum_{i=1}^k \sigma(\hat{y}_i)$$

In [ ]:
def compute_tabm_loss(y_pred, y_true, k=None):
    """
    Computes independent BCE loss for each of the k submodels and averages.
    y_pred shape: (batch_size, k, 1) or (batch_size, k)
    y_true shape: (batch_size,) or (batch_size, 1)
    """
    if y_pred.ndim == 3:
        y_target = y_true.view(-1, 1, 1).expand_as(y_pred)
    elif y_pred.ndim == 2:
        y_target = y_true.view(-1, 1).expand_as(y_pred)
    else:
        y_target = y_true.expand_as(y_pred)
    return F.binary_cross_entropy_with_logits(y_pred, y_target.float())

def predict_tabm_probs(model, loader, device):
    """
    Inference: Sigmoid per submodel, then average across all k ensemble members.
    """
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                x_num, x_cat, _ = batch
            else:
                x_num, x_cat = batch
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            
            y_pred = model(x_num, x_cat)
            probs = torch.sigmoid(y_pred).mean(dim=1).squeeze(-1)
            all_probs.append(probs.cpu().numpy())
            
    return np.concatenate(all_probs)

## Stratified 5-Fold Cross Validation
We train TabM using 5-Fold Stratified Cross Validation:
1. `StandardScaler` is fit on the training fold only to prevent leakage.
2. `TabM.make` creates the BatchEnsemble architecture with $k=32$ ensemble submodels.
3. Optimized using `AdamW` and `CosineAnnealingLR`.
4. Validation **ROC-AUC** is evaluated at each epoch, saving the checkpoint with the highest validation score.
5. Out-of-fold (OOF) predictions and test predictions are stored.

In [ ]:
print("=" * 65)
print(f"Starting {CFG.n_splits}-Fold Cross-Validation for TabM")
print(f"Device: {device} | k={CFG.k} | Batch size: {CFG.batch_size}")
print("=" * 65)

skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

oof_preds = np.zeros(len(train_eng))
test_preds = np.zeros(len(test_eng))
fold_auc_scores = []
fold_log_loss_scores = []

X_test_cat = test_eng[cat_cols].values

for fold, (train_idx, val_idx) in enumerate(skf.split(train_eng, y)):
    print(f"\n{'=' * 20} FOLD {fold + 1} / {CFG.n_splits} {'=' * 20}")
    fold_start_time = time.time()
    
    # Train / Val splits
    X_train_num_raw = train_eng.loc[train_idx, num_cols].values
    X_val_num_raw = train_eng.loc[val_idx, num_cols].values
    X_test_num_raw = test_eng[num_cols].values
    
    X_train_cat = train_eng.loc[train_idx, cat_cols].values
    X_val_cat = train_eng.loc[val_idx, cat_cols].values
    
    y_train_fold = y[train_idx]
    y_val_fold = y[val_idx]
    
    # Feature Scaling
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num_raw)
    X_val_num = scaler.transform(X_val_num_raw)
    X_test_num = scaler.transform(X_test_num_raw)
    
    # DataLoaders
    train_dataset = TabularDataset(X_train_num, X_train_cat, y_train_fold)
    val_dataset = TabularDataset(X_val_num, X_val_cat, y_val_fold)
    test_dataset = TabularDataset(X_test_num, X_test_cat)
    
    num_workers = 2 if torch.cuda.is_available() else 0
    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, 
                              num_workers=num_workers, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size * 2, shuffle=False, 
                            num_workers=num_workers, pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size * 2, shuffle=False, 
                             num_workers=num_workers, pin_memory=torch.cuda.is_available())
    
    # Instantiate TabM Model
    model = TabM.make(
        n_num_features=len(num_cols),
        cat_cardinalities=cat_cardinalities,
        d_out=1,
        k=CFG.k,
        n_blocks=CFG.n_blocks,
        d_block=CFG.d_block,
        dropout=CFG.dropout,
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=1e-6)
    
    best_val_auc = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_path = f"best_tabm_fold{fold}.pt"
    
    for epoch in range(1, CFG.epochs + 1):
        model.train()
        train_loss_accum = 0.0
        
        for x_num_b, x_cat_b, y_b in train_loader:
            x_num_b = x_num_b.to(device, non_blocking=True)
            x_cat_b = x_cat_b.to(device, non_blocking=True)
            y_b = y_b.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            y_pred = model(x_num_b, x_cat_b)
            loss = compute_tabm_loss(y_pred, y_b, CFG.k)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss_accum += loss.item() * len(y_b)
            
        scheduler.step()
        train_loss = train_loss_accum / len(train_dataset)
        
        # Validate
        val_probs = predict_tabm_probs(model, val_loader, device)
        val_auc = roc_auc_score(y_val_fold, val_probs)
        val_ll = log_loss(y_val_fold, val_probs)
        
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:02d}/{CFG.epochs:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_ll:.4f} | Val ROC-AUC: {val_auc:.5f} | LR: {current_lr:.6f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= CFG.patience:
                print(f"[Early Stopping] Stopped at epoch {epoch}. Best epoch was {best_epoch} (AUC: {best_val_auc:.5f})")
                break
                
    # Load best checkpoint
    model.load_state_dict(torch.load(best_model_path))
    
    # Out-of-fold inference
    fold_oof_probs = predict_tabm_probs(model, val_loader, device)
    oof_preds[val_idx] = fold_oof_probs
    
    final_auc = roc_auc_score(y_val_fold, fold_oof_probs)
    final_ll = log_loss(y_val_fold, fold_oof_probs)
    fold_auc_scores.append(final_auc)
    fold_log_loss_scores.append(final_ll)
    
    elapsed = time.time() - fold_start_time
    print(f"--> Fold {fold + 1} Best ROC-AUC: {final_auc:.5f} | Log Loss: {final_ll:.5f} | Elapsed: {elapsed:.1f}s")
    
    # Test inference
    fold_test_probs = predict_tabm_probs(model, test_loader, device)
    test_preds += fold_test_probs / CFG.n_splits
    
    # Memory Cleanup
    del model, optimizer, scheduler, train_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("\n" + "=" * 65)
print("Cross-Validation Completed!")
print("=" * 65)

## Out-Of-Fold Evaluation & Diagnostics
Compute the overall Out-of-Fold (OOF) ROC-AUC score and plot the ROC curve.

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)
overall_ll = log_loss(y, oof_preds)

print("-" * 45)
print(f"OVERALL 5-FOLD OOF ROC-AUC:  {overall_auc:.5f}")
print(f"OVERALL 5-FOLD OOF LOG-LOSS: {overall_ll:.5f}")
print("-" * 45)

for f, (auc_score, ll_score) in enumerate(zip(fold_auc_scores, fold_log_loss_scores)):
    print(f"  Fold {f + 1}: ROC-AUC = {auc_score:.5f} | Log-Loss = {ll_score:.5f}")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="crimson", lw=2, label=f"TabM OOF ROC (AUC = {overall_auc:.5f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Random Chance")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=11)
plt.ylabel("True Positive Rate", fontsize=11)
plt.title("Playground Series s6e10 - TabM ROC Curve", fontsize=13)
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save OOF predictions for blending / ensembling
oof_df = pd.DataFrame({
    CFG.id_col: train_eng[CFG.id_col],
    'target': y,
    'oof_pred_prob': oof_preds
})
oof_df.to_csv('tabm_oof_predictions.csv', index=False)
print("Saved Out-Of-Fold predictions to tabm_oof_predictions.csv")

## Submission Generation
Generate the final test predictions, perform validation checks, and save `submission.csv`.

In [ ]:
submission = pd.DataFrame({
    CFG.id_col: test_ids,
    CFG.target: test_preds
})

submission.to_csv('submission.csv', index=False)
print(f"Submission successfully saved to submission.csv | Total Rows: {len(submission):,}")

# Sanity Checks
print("\n--- Verification Checks ---")
print(f"1. Missing values in predictions: {submission[CFG.target].isnull().sum()}")
print(f"2. Prediction Range: [{submission[CFG.target].min():.4f}, {submission[CFG.target].max():.4f}]")
print(f"3. Prediction Mean:  {submission[CFG.target].mean():.4f}")
print(f"4. Matches sample_submission shape: {submission.shape == sample_sub.shape}")

display(submission.head(10))